# CIFAR-10: attention-head fault tolerance in a ViT-style encoder

Replication of the `transformers_EXP_A` protocol on a **real** image corpus.

## Architecture: Tiny GPT-2, decoder-only

This notebook is a derived copy of the matching notebook in
`notebooks/base_transformer/`. Every dataset, fault-injection and analysis
cell is identical; the only change is the architecture, taken from
`C.TINY_GPT2_PARAM` - the Tiny GPT-2 of *Dropout and the Outliers*
(Hezbri et al., Table 4): **6 blocks x 6 heads = 36 attention heads**,
`d_model` 384, `d_ff` 1536, pre-normalisation, learnable absolute positional
embeddings, **no biases anywhere**, **causal (left-to-right) attention**, GELU.

| | Base family | Tiny ViT family | This family |
|---|---|---|---|
| Layers x heads | 4 x 8 = 32 | 6 x 6 = 36 | **6 x 6 = 36** |
| Width / FFN | 256 / 1024 | 384 / 1536 | **384 / 1536** |
| Normalisation | post-norm | pre-norm | **pre-norm** |
| Positional encoding | sinusoidal | learnable | **learnable** |
| q/k/v biases | present | absent | **absent** |
| Other biases | present | present | **absent** |
| Attention | bidirectional | bidirectional | **causal** |
| Read-out token | CLS at position 0 | CLS at position 0 | **CLS at last position** |
| Stack parameters | 3,159,040 | 10,639,872 | **10,621,440** |

**Against the Tiny ViT family this is very nearly a single-factor contrast.**
The paper gives its Tiny GPT-2 and Tiny ViT the same six-block, six-head,
width-384 shape, so only two things differ here, and the first is nearly free:
dropping the remaining biases removes 18,432 of 10,639,872 stack parameters
(0.17%). The substantive variable is **causal versus bidirectional attention**.
That makes the Tiny ViT -> Tiny GPT-2 comparison a much cleaner attribution
than the base -> Tiny ViT one, where seven choices move at once.

Two consequences of causal masking to keep in mind when reading the results:

1. **The read-out token moved.** Under a causal mask a CLS token at position 0
   could attend only to itself, so it is appended at the end instead and
   `encode()` reads the last position. Everything else in the pipeline is
   position-agnostic.
2. **`max_cumulative_attention` is not comparable across families.**
   `attention_stats.csv` sums each key's attention over all queries; under a
   causal mask key `j` is visible only to queries `j` and later, so its ceiling
   falls from `L` to `L - j` and the maximum over keys is structurally biased
   toward early positions. Compare that column within this family only.
   (`mean_cumulative_attention` is exactly 1.0 in every family by construction
   - row sums of 1 over `L` queries and `L` keys - so it carries no
   information either way.) The criticality, degradation and safe-region
   results are unaffected: they are defined on task metrics, not attention mass.

The training recipe, the fault protocol, the criticality definitions and the
evaluation subsets are deliberately unchanged, so the degradation numbers are
comparable with the other two families run for the same dataset.

The token embedding is replaced by a 4x4 patch projection (8x8 = 64 patch tokens plus CLS); the fault model, criticality definition and safe-region safe-region analysis are unchanged.

**Dataset**: CIFAR-10, 50 000 training / 10 000 test images, 10 classes (`cifar-10-python.tar.gz`, md5 `c58f30108f718f92721af3b95e74349a`). This is the same corpus the Dropout-and-Outliers paper uses for its ViT, so the two studies become directly comparable.

**Model**: fixed *Base* configuration - 4 encoder layers x 8 heads = **32 attention heads**.

A crashed head here removes a channel of *spatial patch* attention rather than syntactic or temporal attention, which makes this the strongest test of whether the layer-position pattern is universal.

**Research questions** (as in the source study)

- **RQ1** Do Transformer models contain critical and redundant attention heads?
- **RQ2** Is degradation explained by the number of failed heads alone, or also by which heads fail?
- **RQ3** How do the fault-tolerance results compare across datasets and modalities, given one fixed architecture?

**Hypotheses**

- **H1** Attention-head criticality is non-uniform across the architecture.
- **H2** Layer-wise fragility is non-monotonic and task-dependent.
- **H3** Degradation increases with the number of failed attention heads.
- **H4** Criticality-aware fault severity adds information beyond the raw fault count.
- **H5** Configurations containing more critical heads are likelier to exceed an acceptable degradation threshold.

## Imports

In [1]:
import json, pathlib, sys, time

import numpy as np
import pandas as pd
import torch

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from faulttol import config as C
from faulttol import multihead, plots, saferegion, single_head
from faulttol.injection import collect_attention_stats_per_head, get_all_heads
from faulttol.runtime import configure_threads, device_report, select_device
from faulttol.training import make_loader

print("torch", torch.__version__, "| threads", configure_threads(),
      "| cuda build", torch.version.cuda, "| cuda available", torch.cuda.is_available())
print("root:", ROOT)

torch 2.14.0+cu126 | threads 8 | cuda build 12.6 | cuda available True
root: D:\Laboratory\Transformers fault tollerance


## Global variables

In [2]:
RESULTS_DIR = ROOT / "results" / "tiny_gpt2" / "cifar10"
MODEL_NAME = "tiny_gpt2_cifar10"
ARCHIVE = ROOT / "data" / "cifar-10-python.tar.gz"
# NVIDIA GPU when one is visible; FT_DEVICE=cpu reproduces the CPU-only run
DEVICE = select_device()

IMAGE_SIZE, PATCH_SIZE, NUM_CLASSES = 32, 4, 10
CLASS_NAMES = ["airplane", "automobile", "bird", "cat", "deer",
               "dog", "frog", "horse", "ship", "truck"]
MEAN = np.array([0.4914, 0.4822, 0.4465], dtype=np.float32)
STD = np.array([0.2470, 0.2435, 0.2616], dtype=np.float32)

MODEL_CONFIG = dict(C.TINY_GPT2_PARAM, activation="gelu")

TRAIN_CONFIG = {
    "loss": "CrossEntropyLoss",
    "optimizer": "AdamW",
    # a from-scratch ViT on 32x32 images needs a larger step and stronger decay
    # than the EXP_A synthetic task; cosine annealing follows the ViT recipe of
    # the Dropout-and-Outliers paper, which uses this same corpus
    "learning_rate": 6e-4,
    "weight_decay": 5e-2,
    "batch_size": 128,
    "epochs": C._env_int("FT_EPOCHS", 40),
    "patience": C._env_int("FT_PATIENCE", 8),
    "grad_clip": 1.0,
    "label_smoothing": 0.1,
    "selection_metric": "roc_auc",
    "cosine_schedule": True,
    "augmentation": "RandomCrop(32, padding=4) + RandomHorizontalFlip",
}

CRIT_COL = "D_pos_mean"
DEG_COL = "D_mean"
SAFE_COL = "D_pos_mean"
DEG_LABEL = "mean probability damage"
SAFE_LABEL = "mean positive damage"
TAUS = C.CLS_TAUS
TAU_LABELS = ["tau_%.3f" % t for t in TAUS]

print("device:", device_report(DEVICE))
print("patches %dx%d = %d tokens | epochs %d"
      % (IMAGE_SIZE // PATCH_SIZE, IMAGE_SIZE // PATCH_SIZE,
         (IMAGE_SIZE // PATCH_SIZE) ** 2, TRAIN_CONFIG["epochs"]))

device: cuda:0 NVIDIA GeForce RTX 4070 Laptop GPU | 8.0 GB | sm_89 | torch cuda 12.6 | tf32 off
patches 8x8 = 64 tokens | epochs 40


In [3]:
OUT = pathlib.Path(RESULTS_DIR)
TABLES, FIGURES, MODELS = OUT / "tables", OUT / "figures", OUT / "models"
for d in (TABLES, FIGURES, MODELS):
    d.mkdir(parents=True, exist_ok=True)

def save(df, name):
    path = TABLES / name
    df.to_csv(path, index=False)
    print("saved", path.relative_to(ROOT), df.shape)
    return path

print("artifacts ->", OUT)
print("K_TRIALS=%d  N_FAULT=%d" % (C.K_TRIALS, C.N_FAULT))

artifacts -> D:\Laboratory\Transformers fault tollerance\experiments\tiny_gpt2\cifar10
K_TRIALS=10000  N_FAULT=1000


## STEP 1: CIFAR-10 dataset

In [4]:
import pickle, tarfile

def load_cifar10():
    train_batches, test_batch = [], None
    with tarfile.open(ARCHIVE, "r:gz") as tar:
        for member in tar.getmembers():
            name = pathlib.PurePosixPath(member.name).name
            if name.startswith("data_batch_"):
                train_batches.append((name, pickle.load(tar.extractfile(member), encoding="bytes")))
            elif name == "test_batch":
                test_batch = pickle.load(tar.extractfile(member), encoding="bytes")
    train_batches.sort(key=lambda kv: kv[0])

    def to_arrays(batch):
        x = batch[b"data"].reshape(-1, 3, 32, 32).astype(np.float32) / 255.0
        y = np.asarray(batch[b"labels"], dtype=np.int64)
        return (x - MEAN[None, :, None, None]) / STD[None, :, None, None], y

    xs, ys = zip(*[to_arrays(b) for _, b in train_batches])
    tx, ty = to_arrays(test_batch)
    return np.concatenate(xs), np.concatenate(ys), tx, ty

train_x, train_y, test_x, test_y = load_cifar10()
print("raw: train=%s test=%s" % (train_x.shape, test_x.shape))
print("class balance:", np.bincount(train_y))

raw: train=(50000, 3, 32, 32) test=(10000, 3, 32, 32)
class balance: [5000 5000 5000 5000 5000 5000 5000 5000 5000 5000]


In [5]:
import torch.nn.functional as F

def augment_batch(xb):
    """RandomCrop(32, padding=4) + RandomHorizontalFlip, batched.

    Vectorised via advanced indexing: with the model on the GPU a per-sample
    Python loop here is the slowest part of the epoch.  The random draws are
    taken in the same order and shape as the loop version, so the augmentation
    stream is unchanged.
    """
    b = xb.size(0)
    padded = F.pad(xb, (4, 4, 4, 4), mode="reflect")
    ox = torch.randint(0, 9, (b,))
    oy = torch.randint(0, 9, (b,))
    flip = torch.rand(b) < 0.5

    ar = torch.arange(32)
    rows = (oy.view(b, 1) + ar.view(1, 32)).view(b, 1, 32, 1)
    # a horizontal flip is the crop read with its columns reversed
    cols = torch.where(flip.view(b, 1), (31 - ar).view(1, 32), ar.view(1, 32))
    cols = (ox.view(b, 1) + cols).view(b, 1, 1, 32)

    return padded[torch.arange(b).view(b, 1, 1, 1),
                  torch.arange(3).view(1, 3, 1, 1),
                  rows, cols]

def balanced_subset(y, n_total, seed):
    rng = np.random.default_rng(seed)
    classes = np.unique(y)
    per = n_total // len(classes)
    picks = [rng.choice(np.flatnonzero(y == c), size=per, replace=False) for c in classes]
    out = np.concatenate(picks)
    rng.shuffle(out)
    return out

rng = np.random.default_rng(C.SPLIT_SEED)
perm = rng.permutation(len(train_x))
n_val = 5_000
val_idx, tr_idx = perm[:n_val], perm[n_val:]

n_sub = C._env_int("FT_TRAIN_SUBSAMPLE", 0)
if n_sub:
    tr_idx = tr_idx[:n_sub]

X_train, y_train = torch.from_numpy(train_x[tr_idx]), train_y[tr_idx]
X_val, y_val = torch.from_numpy(train_x[val_idx]), train_y[val_idx]

X_test_all = torch.from_numpy(test_x)
eval_idx = balanced_subset(test_y, C.N_FAULT, C.FAULT_SPLIT_SEED)
calib_mask = np.ones(len(test_y), dtype=bool)
calib_mask[eval_idx] = False
X_calib, y_calib = X_test_all[calib_mask], test_y[calib_mask]
X_eval, y_eval = X_test_all[eval_idx].contiguous(), test_y[eval_idx]

print("train=%d val=%d calib=%d fault_eval=%d"
      % (len(X_train), len(X_val), len(X_calib), len(X_eval)))
print("fault-eval class balance:", np.bincount(y_eval))

train=45000 val=5000 calib=9000 fault_eval=1000
fault-eval class balance: [100 100 100 100 100 100 100 100 100 100]


## STEP 2: encoder-only Vision Transformer

In [6]:
from faulttol.metrics import (classification_clean_metrics, classification_damage,
                              classification_outputs)
from faulttol.model import ImageClassifier, count_params
from faulttol.training import train_classifier

torch.manual_seed(C.TRAIN_SEED)
np.random.seed(C.TRAIN_SEED)

model = ImageClassifier(MODEL_CONFIG, image_size=IMAGE_SIZE, patch_size=PATCH_SIZE,
                        in_channels=3, num_classes=NUM_CLASSES, activation="gelu")
print("params = %d" % count_params(model))
print("tokens = %d patches (+CLS)" % model.n_patches)
print("heads  = %d" % len(get_all_heads(model)))
model

params = 10669834
tokens = 64 patches (+CLS)
heads  = 36


ImageClassifier(
  (pos): LearnedPositionalEmbedding()
  (layers): ModuleList(
    (0-5): 6 x EncoderLayer(
      (attn): MultiHeadAttention(
        (W_q): Linear(in_features=384, out_features=384, bias=False)
        (W_k): Linear(in_features=384, out_features=384, bias=False)
        (W_v): Linear(in_features=384, out_features=384, bias=False)
        (W_o): Linear(in_features=384, out_features=384, bias=False)
      )
      (ff): Sequential(
        (0): Linear(in_features=384, out_features=1536, bias=False)
        (1): GELU(approximate='none')
        (2): Dropout(p=0.05, inplace=False)
        (3): Linear(in_features=1536, out_features=384, bias=False)
      )
      (norm1): LayerNorm((384,), eps=1e-05, elementwise_affine=True, bias=False)
      (norm2): LayerNorm((384,), eps=1e-05, elementwise_affine=True, bias=False)
      (dropout): Dropout(p=0.05, inplace=False)
    )
  )
  (norm_out): LayerNorm((384,), eps=1e-05, elementwise_affine=True, bias=False)
  (dropout): Dropout(p=0

In [7]:
hist = train_classifier(
    model, X_train, y_train, X_val, y_val, DEVICE,
    lr=TRAIN_CONFIG["learning_rate"], weight_decay=TRAIN_CONFIG["weight_decay"],
    epochs=TRAIN_CONFIG["epochs"], batch_size=TRAIN_CONFIG["batch_size"],
    patience=TRAIN_CONFIG["patience"], grad_clip=TRAIN_CONFIG["grad_clip"],
    ckpt_path=MODELS / "best.pt",
    config={"model": MODEL_CONFIG, "train": TRAIN_CONFIG},
    selection_metric=TRAIN_CONFIG["selection_metric"],
    cosine=TRAIN_CONFIG["cosine_schedule"],
    label_smoothing=TRAIN_CONFIG["label_smoothing"],
    augment=augment_batch,
)
save(pd.DataFrame(hist["history"]), "training_history.csv")
pd.DataFrame(hist["history"])

    epoch  1  loss 1.9038  val_acc 0.4220  val_f1 0.4015  val_auc 0.8584  (38s)


    epoch  2  loss 1.7060  val_acc 0.4714  val_f1 0.4606  val_auc 0.8816  (37s)


    epoch  3  loss 1.6163  val_acc 0.5036  val_f1 0.4975  val_auc 0.8960  (37s)


    epoch  4  loss 1.5553  val_acc 0.5304  val_f1 0.5167  val_auc 0.9077  (38s)


    epoch  5  loss 1.5081  val_acc 0.5548  val_f1 0.5453  val_auc 0.9161  (38s)


    epoch  6  loss 1.4644  val_acc 0.5578  val_f1 0.5544  val_auc 0.9166  (38s)


... (30 outputs left out)

    epoch 37  loss 0.8123  val_acc 0.7562  val_f1 0.7537  val_auc 0.9649  (38s)


    early stopping at epoch 37


saved experiments\tiny_gpt2\cifar10\tables\training_history.csv (37, 8)


,epoch,train_loss,secs,val_accuracy,val_precision,val_recall,val_f1,val_roc_auc
0,1,1.903792,37.627171,0.4220,0.438381,0.423945,0.401521,0.858410
1,2,1.705965,37.376807,0.4714,0.498141,0.473606,0.460627,0.881580
2,3,1.616285,37.470271,0.5036,0.516855,0.505028,0.497545,0.895986
3,4,1.555276,37.551086,0.5304,0.535409,0.528322,0.516695,0.907663
4,5,1.508086,37.573854,0.5548,0.559493,0.552245,0.545308,0.916100
5,6,1.464371,37.538021,0.5578,0.587777,0.553910,0.554429,0.916585
6,7,1.432546,37.548648,0.5634,0.581121,0.565258,0.557734,0.920531
7,8,1.400693,37.567490,0.5934,0.598145,0.593548,0.585487,0.927576
8,9,1.373806,37.598807,0.6108,0.613434,0.609796,0.597967,0.934519
9,10,1.346847,37.604868,0.6192,0.616711,0.618680,0.612186,0.934070


In [8]:
calib_metrics = classification_clean_metrics(
    classification_outputs(model, X_calib, y_calib, DEVICE))

def evaluate():
    return classification_outputs(model, X_eval, y_eval, DEVICE, chunk=500)

damage_fn = classification_damage
clean_eval = classification_clean_metrics(evaluate())

t0 = time.time()
for _ in range(5):
    evaluate()
per_eval = (time.time() - t0) / 5.0

print("calibration set (n=%d):" % len(X_calib), calib_metrics)
print("fixed eval subset (n=%d):" % len(X_eval), clean_eval)
print("cost: %.3f s per configuration -> %.1f min for K=%d"
      % (per_eval, per_eval * C.K_TRIALS / 60.0, C.K_TRIALS))

attn_stats = collect_attention_stats_per_head(
    model, make_loader(X_eval, torch.as_tensor(y_eval), 100, shuffle=False), DEVICE)
save(pd.DataFrame(attn_stats), "attention_stats.csv")

EXTRA_META = {
    "dataset": "CIFAR-10", "class_names": CLASS_NAMES, "patch_size": PATCH_SIZE,
    "n_tokens": int(model.n_patches), "train_config": TRAIN_CONFIG,
    "model_config": MODEL_CONFIG, "calibration_clean_metrics": calib_metrics,
    "n_params": count_params(model), "best_val_score": hist["best_score"],
    "seconds_per_evaluation": per_eval,
}

calibration set (n=9000): {'accuracy': 0.7425555555555555, 'precision': 0.7409938213749259, 'recall': 0.7425555555555556, 'f1': 0.7401520643844955, 'roc_auc': 0.9643642935528121}
fixed eval subset (n=1000): {'accuracy': 0.749, 'precision': 0.7477429780014055, 'recall': 0.7489999999999999, 'f1': 0.7463719176881505, 'roc_auc': 0.9639444444444443}
cost: 0.272 s per configuration -> 45.3 min for K=10000


saved experiments\tiny_gpt2\cifar10\tables\attention_stats.csv (36, 5)


## Fault model

**Definition 1 (attention-head crash).** A head is crashed when its output tensor is
replaced with zeros *after* the attention weights multiply V and *before* head
concatenation and the output projection `W_o`.

Faults are injected **at inference only** - the model is trained clean, then frozen.
After every faulted evaluation all masks are restored to 1.

## STEP 3: single-head crash

Every head is crashed in turn while all others stay alive, so each clean/fault comparison differs in exactly one head.

In [9]:
sh_df = single_head.run_single_head_analysis(
    model=model,
    evaluate=evaluate,
    damage_fn=damage_fn,
    criticality_col=CRIT_COL,
    model_name=MODEL_NAME,
    verbose=True,
)
save(sh_df, "single_head_damage.csv")
sh_df.nlargest(10, "criticality")[["head_name", "criticality", "redundancy", DEG_COL]]

  L0_H0    D_pos_mean = 0.136588


  L0_H1    D_pos_mean = 0.050723


  L0_H2    D_pos_mean = 0.060145


  L0_H3    D_pos_mean = 0.144655


  L0_H4    D_pos_mean = 0.169316


  L0_H5    D_pos_mean = 0.115769


... (28 outputs left out)

  L5_H4    D_pos_mean = 0.017257


  L5_H5    D_pos_mean = 0.016971


saved experiments\tiny_gpt2\cifar10\tables\single_head_damage.csv (36, 30)


,head_name,criticality,redundancy,D_mean
4,L0_H4,0.169316,5.906164e-12,0.141601
3,L0_H3,0.144655,1.456519e-01,0.103536
0,L0_H0,0.136588,1.932979e-01,0.103767
5,L0_H5,0.115769,3.162570e-01,0.081508
2,L0_H2,0.060145,6.447775e-01,0.031321
1,L0_H1,0.050723,7.004245e-01,0.023329
22,L3_H4,0.030696,8.187078e-01,0.013632
8,L1_H2,0.029121,8.280079e-01,0.007871
21,L3_H3,0.029100,8.281318e-01,0.010818
18,L3_H0,0.028137,8.338177e-01,0.011787


### Layer fragility and vulnerability map

In [10]:
layer_summary = single_head.build_layer_summary(sh_df)
model_summary = single_head.build_model_summary(sh_df, layer_summary, MODEL_NAME, clean_eval)
save(layer_summary, "layer_summary.csv")

plots.vulnerability_map(sh_df, "%s - single-head vulnerability (%s)" % (MODEL_NAME, CRIT_COL),
                        FIGURES / "vulnerability_map.png")
plots.criticality_profile(sh_df, "%s - criticality concentration" % MODEL_NAME,
                          FIGURES / "criticality_profile.png", ylabel=CRIT_COL)

print("most critical head      :", model_summary["most_critical_head"],
      "(criticality %.6f)" % model_summary["max_criticality"])
print("median / max criticality: %.4f" % (model_summary["median_criticality"] / model_summary["max_criticality"]))
print("mean redundancy         : %.4f" % model_summary["mean_redundancy"])
print("fragile layer by mean   :", model_summary["most_fragile_layer_by_mean"])
print("fragile layer by max    :", model_summary["most_fragile_layer_by_max"])
layer_summary

saved experiments\tiny_gpt2\cifar10\tables\layer_summary.csv (6, 7)


most critical head      : L0_H4 (criticality 0.169316)
median / max criticality: 0.1289
mean redundancy         : 0.7874
fragile layer by mean   : 0
fragile layer by max    : 0


,layer,mean_layer_criticality,max_head_criticality,min_head_criticality,mean_redundancy,n_heads,most_critical_head
0,0,0.112866,0.169316,0.050723,0.333401,6,L0_H4
1,1,0.021431,0.029121,0.014717,0.873429,6,L1_H2
2,2,0.021240,0.027738,0.015739,0.874556,6,L2_H5
3,3,0.022565,0.030696,0.013674,0.866727,6,L3_H4
4,4,0.017604,0.025717,0.013241,0.896027,6,L4_H4
5,5,0.020240,0.028060,0.015392,0.880462,6,L5_H2


## STEP 4: criticality, redundancy and severity weights

`C = mean(max(damage,0))`, `R = 1 - C/max(C)`, `W = C/mean(C)`. Critical heads are the top 10% by criticality.

In [11]:
weight_lookup = single_head.build_head_weight_lookup(sh_df)
critical_heads = single_head.critical_head_set(sh_df, C.CRITICAL_TOP_FRACTION)
print("critical heads (top %d%%): %s"
      % (100 * C.CRITICAL_TOP_FRACTION, ["L%d_H%d" % h for h in critical_heads]))
print("heads with near-zero effect:", int((sh_df["criticality"] <= 1e-6).sum()), "of", len(sh_df))
save(sh_df[["head_name", "layer", "head", "criticality", "redundancy", "severity_weight"]],
     "head_weights.csv")
sh_df.nsmallest(5, "criticality")[["head_name", "criticality", "redundancy", DEG_COL]]

critical heads (top 10%): ['L0_H4', 'L0_H3', 'L0_H0', 'L0_H5']
heads with near-zero effect: 0 of 36
saved experiments\tiny_gpt2\cifar10\tables\head_weights.csv (36, 6)


,head_name,criticality,redundancy,D_mean
26,L4_H2,0.013241,0.921798,0.001953
24,L4_H0,0.013490,0.920326,0.003976
23,L3_H5,0.013674,0.919242,0.003550
27,L4_H3,0.014312,0.915473,0.004938
11,L1_H5,0.014717,0.913080,0.001308


## STEP 5: random multi-head fault injection

`N_fail ~ Uniform{1..32}`, heads drawn without replacement and crashed simultaneously, evaluated on the fixed subset.

In [12]:
n_heads = len(get_all_heads(model))
mh_df = multihead.run_random_multihead(
    model=model,
    evaluate=evaluate,
    damage_fn=damage_fn,
    weight_lookup=weight_lookup,
    critical_heads=critical_heads,
    model_name=MODEL_NAME,
    k_trials=C.K_TRIALS,
    seed=C.MULTIHEAD_FAULT_SEED,
    num_layers=model.num_layers,
)
save(mh_df, "random_multihead.csv")
mh_df[["N_fail", "N_crit_fail", "W_fail", "W_fail_mean", DEG_COL]].describe()

    trial 500/10000  3.6 trials/s  eta 44.4 min


    trial 1000/10000  3.6 trials/s  eta 42.0 min


    trial 1500/10000  3.6 trials/s  eta 39.7 min


    trial 2000/10000  3.6 trials/s  eta 37.3 min


    trial 2500/10000  3.6 trials/s  eta 35.0 min


    trial 3000/10000  3.6 trials/s  eta 32.7 min


... (12 outputs left out)

    trial 9500/10000  3.6 trials/s  eta 2.3 min


    trial 10000/10000  3.6 trials/s  eta 0.0 min


saved experiments\tiny_gpt2\cifar10\tables\random_multihead.csv (10000, 38)


,N_fail,N_crit_fail,W_fail,W_fail_mean,D_mean
count,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000
mean,18.395200,2.045500,18.392193,0.997404,0.340849
std,10.418429,1.401794,10.785454,0.336552,0.181517
min,1.000000,0.000000,0.367896,0.367896,0.000886
25%,9.000000,1.000000,8.974523,0.852285,0.177981
50%,18.000000,2.000000,18.358047,1.000000,0.397478
75%,27.000000,3.000000,27.743864,1.093286,0.513200
max,36.000000,4.000000,36.000000,4.704414,0.564460


### Degradation by fault fraction and failed-head criticality

In [13]:
grid = multihead.degradation_grid(mh_df, DEG_COL)
save(grid.reset_index(), "degradation_grid.csv")
plots.degradation_grid_plot(
    grid, "%s - mean %s by fault fraction and criticality" % (MODEL_NAME, DEG_LABEL),
    FIGURES / "degradation_grid.png", value_label=DEG_LABEL)
grid

saved experiments\tiny_gpt2\cifar10\tables\degradation_grid.csv (5, 5)


crit_quartile,Q1,Q2,Q3,Q4
fault_bin,,,,
0-10%,0.012662,0.033696,0.034142,0.117463
10-25%,0.058078,0.127133,0.156933,0.212533
25-50%,0.190106,0.268399,0.326328,0.391921
50-75%,0.359256,0.430177,0.479001,0.504272
75-100%,0.480664,0.511047,0.529125,0.532715


### How well does fault severity explain degradation?

Rank correlations between each fault-severity descriptor and the degradation, plus the `W_fail` / `N_fail` confound.

In [14]:
from scipy.stats import kendalltau, pearsonr, spearmanr

def severity_correlations(df, deg_col):
    rows = []
    for sev in ("N_fail", "L_fault", "N_crit_fail", "W_fail", "W_fail_mean"):
        x, y = df[sev].to_numpy(float), df[deg_col].to_numpy(float)
        pr, pp = pearsonr(x, y)
        sr, sp = spearmanr(x, y)
        kr, kp = kendalltau(x, y)
        rows.append({"severity": sev, "degradation": deg_col, "pearson": pr,
                     "spearman": sr, "kendall": kr, "spearman_p": sp})
    return pd.DataFrame(rows)

corr_df = severity_correlations(mh_df, DEG_COL)
save(corr_df, "severity_correlations.csv")

# W_fail is partly a restatement of N_fail - report the confound explicitly
conf = spearmanr(mh_df["W_fail"], mh_df["N_fail"]).statistic
print("Spearman(W_fail, N_fail) = %.3f  <- how much of W_fail is just fault count" % conf)
corr_df

saved experiments\tiny_gpt2\cifar10\tables\severity_correlations.csv (5, 6)
Spearman(W_fail, N_fail) = 0.969  <- how much of W_fail is just fault count


,severity,degradation,pearson,spearman,kendall,spearman_p
0,N_fail,D_mean,0.901707,0.915169,0.758689,0.0
1,L_fault,D_mean,0.750646,0.733285,0.604512,0.0
2,N_crit_fail,D_mean,0.909970,0.931751,0.812997,0.0
3,W_fail,D_mean,0.951367,0.969526,0.859142,0.0
4,W_fail_mean,D_mean,0.266913,0.433395,0.309139,0.0


## STEP 6: empirical probabilistic fault-tolerance regions

Widest `(n, c)` with `P(Y <= tau | N_fail <= n, N_crit_fail <= c) >= 0.95`.

In [15]:
grids, safe_summary = saferegion.run_safe_region_analysis(
    mh_df, SAFE_COL, TAUS, n_heads, len(critical_heads),
    min_configs=C.MIN_CONFIGS_PER_BOUNDARY, safe_prob=C.SAFE_PROB,
    tau_labels=TAU_LABELS)
save(grids, "safe_region_grid.csv")
save(safe_summary, "safe_region_summary.csv")

tau_df = saferegion.required_tau_table(
    mh_df, SAFE_COL, n_heads, len(critical_heads),
    min_configs=C.MIN_CONFIGS_PER_BOUNDARY, safe_prob=C.SAFE_PROB)
save(tau_df, "required_tau.csv")
plots.required_tau_plot(
    tau_df, "%s - required degradation threshold for 95%% safety" % MODEL_NAME,
    FIGURES / "required_tau.png", ylabel="tau required (%s)" % SAFE_LABEL)

safe_summary

saved experiments\tiny_gpt2\cifar10\tables\safe_region_grid.csv (540, 9)
saved experiments\tiny_gpt2\cifar10\tables\safe_region_summary.csv (3, 9)
saved experiments\tiny_gpt2\cifar10\tables\required_tau.csv (180, 6)


,tau,safe_boundary,n_fail_max,n_crit_fail_max,max_p_safe,coverage_percent,smallest_required_tau,n_boundaries_evaluated,threshold_name
0,0.010,None,None,None,0.000000,0.0,0.050723,180,tau_0.010
1,0.025,None,None,None,0.710526,0.0,0.050723,180,tau_0.025
2,0.050,None,None,None,0.943609,0.0,0.050723,180,tau_0.050


## STEP 7: summary

In [16]:
summary = {
    "model_name": MODEL_NAME,
    "n_heads": n_heads,
    "n_layers": model.num_layers,
    "k_trials": int(C.K_TRIALS),
    "n_fault_eval_samples": int(C.N_FAULT),
    "clean_metrics": clean_eval,
    "single_head": model_summary,
    "critical_heads": ["L%d_H%d" % h for h in critical_heads],
    "criticality_col": CRIT_COL,
    "degradation_col": DEG_COL,
    "safe_degradation_col": SAFE_COL,
    "severity_correlations": corr_df.to_dict(orient="records"),
    "safe_regions": safe_summary.to_dict(orient="records"),
    "criticality_profile": sorted(sh_df["criticality"].tolist(), reverse=True),
    "median_over_max_criticality": float(sh_df["criticality"].median() / (sh_df["criticality"].max() + 1e-12)),
    "n_heads_zero_effect": int((sh_df["criticality"] <= 1e-6).sum()),
    "worst_single_head": {
        "head": sh_df.loc[sh_df["criticality"].idxmax(), "head_name"],
        "criticality": float(sh_df["criticality"].max()),
        "degradation": float(sh_df.loc[sh_df["criticality"].idxmax(), DEG_COL]),
        "safe_degradation": float(sh_df.loc[sh_df["criticality"].idxmax(), SAFE_COL]),
    },
    **EXTRA_META,
}
(OUT / "summary.json").write_text(json.dumps(summary, indent=2, default=str), encoding="utf-8")
print("wrote", (OUT / "summary.json").relative_to(ROOT))

print()
print("=" * 70)
print("CONCLUSIONS -", MODEL_NAME)
print("=" * 70)
print("clean performance      :", clean_eval)
print("worst single head      : %s  ->  %s = %.6f"
      % (summary["worst_single_head"]["head"], DEG_COL, summary["worst_single_head"]["degradation"]))
print("median/max criticality : %.4f  (1.0 = uniform, 0.0 = fully concentrated)"
      % summary["median_over_max_criticality"])
print("critical heads         :", summary["critical_heads"])
print("layer of critical heads:", sorted({h.split("_")[0] for h in summary["critical_heads"]}))
for r in summary["safe_regions"]:
    print("safe region tau=%-10s: %s (P=%.3f, coverage %.2f%%)"
          % (r.get("threshold_name"), r.get("safe_boundary"), r.get("max_p_safe") or float("nan"),
             r.get("coverage_percent") or 0.0))

wrote experiments\tiny_gpt2\cifar10\summary.json

CONCLUSIONS - tiny_gpt2_cifar10
clean performance      : {'accuracy': 0.749, 'precision': 0.7477429780014055, 'recall': 0.7489999999999999, 'f1': 0.7463719176881505, 'roc_auc': 0.9639444444444443}
worst single head      : L0_H4  ->  D_mean = 0.141601
median/max criticality : 0.1289  (1.0 = uniform, 0.0 = fully concentrated)
critical heads         : ['L0_H4', 'L0_H3', 'L0_H0', 'L0_H5']
layer of critical heads: ['L0']
safe region tau=tau_0.010 : None (P=nan, coverage 0.00%)
safe region tau=tau_0.025 : None (P=0.711, coverage 0.00%)
safe region tau=tau_0.050 : None (P=0.944, coverage 0.00%)
